In [ ]:
"""
Script: train_flowers_3d.py

Description:
    Loads 3D knot singularity CSV datasets, constructs a PyTorch Dataset and DataLoader,
    initializes a 3D convolutional classifier (Classifier3D), trains it using mixed precision,
    tracks training loss, and saves the trained model checkpoint.

Sections:
    1. Imports & Device Setup
    2. Configuration: resolution & hyperparameters
    3. Dataset Loading: deserialize 3D knot CSV files
    4. DataLoader Preparation
    5. Model Initialization & Summary
    6. Training Utilities: loops & plotting
    7. Training Loop
    8. Model Saving
"""

In [ ]:
# -----------------------------------------------------------------------------
# 1. Imports & Device Setup
# -----------------------------------------------------------------------------
import sys
sys.path.append('../')  # Ensure project modules are on PYTHONPATH

import time                       # Track training duration
import itertools                  # Generate knot labels
import json, csv                  # For reading and writing JSON/CSV data

import numpy as np                # Numerical operations
import torch                      # Core PyTorch library
import torch.nn as nn             # Neural network modules
import torch.nn.functional as F   # Activation & loss functions
from torch.optim import Adam      # Optimizer
from torch.optim.lr_scheduler import ReduceLROnPlateau  # LR scheduler
from torch.cuda.amp import autocast, GradScaler         # Mixed precision
from torch.utils.data import TensorDataset, DataLoader  # Data handling
from tqdm import trange           # Progress bars
from torchsummary import summary  # Model summary

from extra_functions_package.all_knots_functions import *  # Knot utilities
from classifier_models import Classifier3D                # 3D CNN model definition

# Use GPU if available, else CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# -----------------------------------------------------------------------------
# 2. Configuration: resolution & hyperparameters
# -----------------------------------------------------------------------------
desired_res = (32, 32, 32)  # Target dimensions (D,H,W) for 3D input

hyperparams = {
    'learning_rate': 1e-5,   # Initial learning rate
    'patience': 0,           # LR scheduler patience epochs
    'decay_epoch': 25,       # Epoch to apply manual LR step
    'factor': 0.2,           # LR decay factor
    'batch_size': 64         # Training batch size
}
num_epochs = 50             # Total training epochs
print_every = 1             # Print loss every N epochs

# Define convolutional stages and pooling configs for the model
stages = [
    [(1, 32, 3, 1, 1), (32, 32, 3, 1, 1), (32, 32, 3, 1, 1)],
    [(32, 64, 5, 1, 1), (64, 64, 5, 1, 1), (64, 64, 5, 1, 1)]
]
pooling_configs = [(2, 2, 1), (2, 2, 1)]

In [ ]:
# -----------------------------------------------------------------------------
# 3. Dataset Loading: deserialize 3D knot CSV files
# -----------------------------------------------------------------------------
# Generate all 4-foil knot labels (e.g. '0000', '0001', ...)
foils = list(itertools.product(range(3), repeat=4))
knots = [''.join(map(str, foil)) for foil in foils]
knot_types = {knot: idx for idx, knot in enumerate(knots)}

# Folders containing CSV data for different Rytov and sample settings
folders = [
    '../4foils_L270_0.15_50_64x64x64_v1',
    '../4foils_L270_0.05_50_64x64x64_v1',
    '../4foils_L270_0.25_50_64x64x64_v1',
    '../4foils_L270_0.15_450_64x64x64_v1',
    '../4foils_L270_0.05_450_64x64x64_v1',
    '../4foils_L270_0.25_450_64x64x64_v1',
]
num_classes = len(knots)

X_list, Y_list = [], []
csv.field_size_limit(10_000_000)  # Allow large JSON entries
flag_print_shape = True  # Only print the input shape once

for folder in folders:
    for knot in knots:
        path = f"{folder}/data_{knot}.csv"
        try:
            with open(path, 'r') as f:
                reader = csv.reader(f)
                for row in reader:
                    # Each row: JSON-encoded list [idx, (Nx,Ny,Nz), x,y,z points...]
                    arr = json.loads(row[0])
                    points = np.array(arr[2:], dtype=int)
                    Nx, Ny, Nz = arr[1]

                    # Print shape once for verification
                    if flag_print_shape:
                        print(f"Loaded grid shape: ({Nx},{Ny},{Nz})")
                        flag_print_shape = False

                    # Rescale coordinates if needed
                    if (Nx, Ny, Nz) != desired_res:
                        scales = np.array(desired_res) / np.array([Nx, Ny, Nz])
                        points = np.rint(points * scales).astype(int)

                    # Build binary voxel grid
                    voxels = np.zeros(desired_res, dtype=int)
                    for x, y, z in points:
                        if 0 <= x < desired_res[0] and 0 <= y < desired_res[1] and 0 <= z < desired_res[2]:
                            voxels[x, y, z] = 1

                    X_list.append(voxels)
                    Y_list.append(knot_types[knot])
        except FileNotFoundError:
            print(f"File not found: {path}")
        except json.JSONDecodeError:
            print(f"Error decoding JSON: {path}")

print(f"Total samples loaded: {len(X_list)} ({len(X_list)//num_classes} per class)")

In [ ]:
# -----------------------------------------------------------------------------
# 4. DataLoader Preparation
# -----------------------------------------------------------------------------
X_np = np.stack(X_list)                # [N, D, H, W]
y_np = np.array(Y_list)

# Convert to tensors and one-hot encode labels
X_tensor = torch.tensor(X_np).unsqueeze(1).float()  # [N,1,D,H,W]
y_tensor = F.one_hot(torch.tensor(y_np), num_classes).float()

# Create TensorDataset and DataLoader
dataset = TensorDataset(X_tensor, y_tensor)
dataloader = DataLoader(dataset, batch_size=hyperparams['batch_size'], shuffle=True)

In [ ]:
# -----------------------------------------------------------------------------
# 5. Model Initialization & Summary
# -----------------------------------------------------------------------------
model = Classifier3D(stages, pooling_configs, num_classes=num_classes).to(device)
model.initialize_weights()  # Apply custom weight init

# Display model architecture and parameter counts
summary(model, input_size=X_tensor.shape[1:])


In [ ]:
# -----------------------------------------------------------------------------
# 6. Training Utilities: loops & plotting
# -----------------------------------------------------------------------------
# Loss, optimizer, scheduler, and mixed-precision scaler
criterion = nn.CrossEntropyLoss().to(device)
optimizer = Adam(model.parameters(), lr=hyperparams['learning_rate'])
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=hyperparams['factor'],
                              patience=hyperparams['patience'], verbose=True)
scaler = GradScaler()


def loop_train(model, loader):
    """One epoch training loop with mixed precision."""
    model.train()
    total_loss = 0.0
    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        with autocast():
            outputs = model(inputs)
            loss = criterion(outputs, torch.argmax(targets, dim=1))
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
    return total_loss / len(loader)


def plot_losses(losses):
    """Plot training loss curve."""
    plt.figure()
    plt.plot(losses, label='Train Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.show()

In [ ]:
# -----------------------------------------------------------------------------
# 7. Training Loop
# -----------------------------------------------------------------------------
train_losses = []
start_time = time.time()
scheduler.step(float('inf'))  # Initialize scheduler
for epoch in trange(num_epochs, desc='Epochs'):
    loss = loop_train(model, dataloader)
    train_losses.append(loss)

    # Manual LR decay at specified epoch
    if epoch == hyperparams['decay_epoch'] - 1:
        scheduler.step(loss)

    if (epoch + 1) % print_every == 0:
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {loss:.4f}")

total_time = time.time() - start_time
print(f"Training completed in {total_time:.1f}s")
plot_losses(train_losses)

In [ ]:
# -----------------------------------------------------------------------------
# 8. Model Saving
# -----------------------------------------------------------------------------
checkpoint = {
    'model_state_dict': model.state_dict(),
    'hyperparams': hyperparams,
    'num_classes': num_classes,
    'stages': stages,
    'pooling_configs': pooling_configs,
    'desired_res': desired_res,
}
torch.save(checkpoint, 'classifier_4foil_3d.pth')
print("Model saved to classifier_4foil_3d.pth")
